<a href="https://colab.research.google.com/github/diegovc1987-boop/Ejercicio-4/blob/main/Ejercicio4_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!pip install gradio transformers google-generativeai deep-translator

### Setup and API Key Configuration

We'll import the necessary libraries and configure the Gemini API key. Make sure your `GEMINI_API_KEY` is set in Colab's 'Secrets' panel (the key icon on the left sidebar).

In [19]:
import gradio as gr
from transformers import pipeline
import google.generativeai as genai
from deep_translator import GoogleTranslator
from google.colab import userdata
import os

# Configure Gemini API Key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    # Switching to the 2.0 Flash Lite model for the lowest possible latency
    gemini_model = genai.GenerativeModel('gemini-2.0-flash-lite-001')
else:
    print("GOOGLE_API_KEY not found.")
    gemini_model = None

# Sentiment pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Sentiment Analysis Function

This function uses a pre-trained HuggingFace model to determine the sentiment (positive or negative) of the input text.

In [5]:
def analyze_sentiment(text):
    if not text:
        return "Please enter some text."
    if not sentiment_pipeline:
        return "Sentiment analysis model not loaded."
    try:
        # Translate text to English before sentiment analysis
        translator = GoogleTranslator(source='auto', target='en')
        translated_text = translator.translate(text)

        result = sentiment_pipeline(translated_text[:512])[0]  # Truncate to 512 tokens
        sentiment = "POSITIVO 😊" if result["label"] == "POSITIVE" else "NEGATIVO 😞"
        score = round(result["score"], 4)
        return f"Sentiment: {sentiment} (Confidence: {score*100:.2f}%) - Original: '{text}', Translated: '{translated_text}'"
    except Exception as e:
        return f"Error in sentiment analysis: {e}"

sentiment_interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter text for sentiment analysis..."),
    outputs="text",
    title="📊 Sentiment Analyzer",
    description="Analyze the sentiment of your text (Positive or Negative)."
)

### Text Summarization Function

This function uses the Gemini model to generate a concise summary of the provided text.

In [18]:
def summarize_text(text):
    if not text:
        return "Por favor, ingresa un texto."
    if not gemini_model:
        return "Gemini no está configurado."

    try:
        prompt = f"Resume este texto en español de forma muy breve (máximo 2 frases):\n\n{text}"
        # Direct call for maximum speed
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

summarization_interface = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(lines=3, placeholder="Escribe aquí..."),
    outputs="text",
    title="📝 Resumidor Ultra-Rápido",
    description="Resumen instantáneo usando Gemini 2.0 Flash Lite."
)

### Translation Function

This function leverages `deep_translator` to translate text to a specified target language.

In [12]:
def translate_text(text, target_lang="es"):
    if not text:
        return "Please enter some text."
    try:
        translator = GoogleTranslator(source='auto', target=target_lang)
        result = translator.translate(text[:5000])  # Limit character count
        return result
    except Exception as e:
        return f"Error in translation: {e}"

translation_interface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter text to translate..."),
        gr.Dropdown(
            choices=["es", "en", "fr", "de", "it", "pt", "ja", "zh-CN"],
            label="Target Language",
            value="es"
        )
    ],
    outputs="text",
    title="🌍 Translator",
    description="Translate text to various languages."
)

### Launch the Gradio Multi-Tool Application

Finally, we combine all three interfaces into a single `gr.TabbedInterface` and launch the application. The public URL will be displayed below after execution.

In [13]:
import asyncio
import time

async def direct_gemini_test_with_timeout():
    print("Starting isolated Gemini API test (using asyncio.to_thread for synchronous call)...")
    test_prompt = "Hello, what is your name?"

    try:
        start_time = time.time()
        print(f"[{time.time() - start_time:.2f}s] Attempting gemini_model.generate_content in a thread...")
        # Run the synchronous generate_content in a separate thread
        response = await asyncio.to_thread(gemini_model.generate_content, test_prompt)
        end_time = time.time()
        print(f"[{end_time - start_time:.2f}s] Received response: {response.text}")
    except Exception as e:
        end_time = time.time()
        print(f"[{end_time - start_time:.2f}s] ERROR: An unexpected error occurred: {e}")
    print("Finished isolated Gemini API test.")

await direct_gemini_test_with_timeout()

Starting isolated Gemini API test (using asyncio.to_thread for synchronous call)...
[0.00s] Attempting gemini_model.generate_content in a thread...
[2.47s] Received response: Hello! I am Gemini, a large language model built by Google. How can I help you today?
Finished isolated Gemini API test.


In [14]:
import google.generativeai as genai

print("Listing available Gemini models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  Model: {m.name}, Description: {m.description}")

Listing available Gemini models:
  Model: models/gemini-2.5-flash, Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
  Model: models/gemini-2.5-pro, Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
  Model: models/gemini-2.0-flash, Description: Gemini 2.0 Flash
  Model: models/gemini-2.0-flash-001, Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
  Model: models/gemini-2.0-flash-lite-001, Description: Stable version of Gemini 2.0 Flash-Lite
  Model: models/gemini-2.0-flash-lite, Description: Gemini 2.0 Flash-Lite
  Model: models/gemini-2.5-flash-preview-tts, Description: Gemini 2.5 Flash Preview TTS
  Model: models/gemini-2.5-pro-preview-tts, Description: Gemini 2.5 Pro Preview TTS
  Model: models/gemma-4-26b-a4b-it, Description: Gemma 4 26B A4B IT
  Model: models/gemma-4-31b-it

Please run the above cell and share the output. This will tell us the correct model names to use for `gemini_model = genai.GenerativeModel(...)`.

Please share the complete output of this cell after you run it. This will tell us definitively if the API call itself is able to complete or if it's consistently hanging/timing out.

In [ ]:
multi_tool_app = gr.TabbedInterface(
    [sentiment_interface, summarization_interface, translation_interface],
    ["Sentiment Analyzer", "Text Summarizer", "Translator"],
    title="AI Multi-Tool (Gradio)"
)

# Added debug=True to see real-time errors in the notebook if the timeout persists
multi_tool_app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fba83e7b46a65ceb19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Task
Prepare the files and instructions for deploying the AI Multi-Tool Gradio application to Hugging Face Spaces. Follow these steps:

1.  **Create `requirements.txt`**: Generate a file containing the necessary dependencies: `gradio`, `transformers`, `google-generativeai`, and `deep-translator`.
2.  **Create `app.py`**: Consolidate all the logic from the notebook into a single production-ready script. This includes imports, the configuration for `gemini-2.0-flash-lite-001` (retrieving the API key from environment variables via `os.getenv('GOOGLE_API_KEY')`), the sentiment analysis, summarization, and translation functions, and the Gradio `TabbedInterface` setup.
3.  **Provide Deployment Instructions**: Outline the steps to create a new Hugging Face Space (using the Gradio SDK), upload the generated files, and securely configure the `GOOGLE_API_KEY` in the Space's "Variables and Secrets" settings.

## Preparar el archivo de requisitos

### Subtask:
Crear un archivo llamado 'requirements.txt' que contenga todas las librerías necesarias para el despliegue.


**Reasoning**:
I will create a 'requirements.txt' file containing the necessary libraries for the deployment as specified in the instructions.



In [23]:
%%writefile requirements.txt
gradio
transformers
google-generativeai
deep-translator

Writing requirements.txt


## Preparar el archivo principal de la aplicación

### Subtask:
Generar el archivo 'app.py' consolidando todo el código actual en un solo script listo para producción.


**Reasoning**:
I will create the 'app.py' file by consolidating the notebook logic into a single script, ensuring it's ready for production deployment on Hugging Face Spaces.



In [22]:
%%writefile app.py
import gradio as gr
from transformers import pipeline
import google.generativeai as genai
from deep_translator import GoogleTranslator
import os

# Configure Gemini API Key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.0-flash-lite-001')
else:
    gemini_model = None

# Sentiment pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

def analyze_sentiment(text):
    if not text:
        return "Please enter some text."
    try:
        translator = GoogleTranslator(source='auto', target='en')
        translated_text = translator.translate(text)
        result = sentiment_pipeline(translated_text[:512])[0]
        sentiment = "POSITIVO 😊" if result["label"] == "POSITIVE" else "NEGATIVO 😞"
        score = round(result["score"], 4)
        return f"Sentiment: {sentiment} (Confidence: {score*100:.2f}%) - Original: '{text}', Translated: '{translated_text}'"
    except Exception as e:
        return f"Error in sentiment analysis: {e}"

def summarize_text(text):
    if not text:
        return "Por favor, ingresa un texto."
    if not gemini_model:
        return "Gemini no está configurado. Verifica la API Key."
    try:
        prompt = f"Resume este texto en español de forma muy breve (máximo 2 frases):\n\n{text}"
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

def translate_text(text, target_lang="es"):
    if not text:
        return "Please enter some text."
    try:
        translator = GoogleTranslator(source='auto', target=target_lang)
        result = translator.translate(text[:5000])
        return result
    except Exception as e:
        return f"Error in translation: {e}"

# Create Interfaces
sentiment_interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter text for sentiment analysis..."),
    outputs="text",
    title="📊 Sentiment Analyzer"
)

summarization_interface = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(lines=3, placeholder="Escribe aquí..."),
    outputs="text",
    title="📝 Resumidor Ultra-Rápido"
)

translation_interface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter text to translate..."),
        gr.Dropdown(
            choices=["es", "en", "fr", "de", "it", "pt", "ja", "zh-CN"],
            label="Target Language",
            value="es"
        )
    ],
    outputs="text",
    title="🌍 Translator"
)

# Combine into Tabbed Interface
multi_tool_app = gr.TabbedInterface(
    [sentiment_interface, summarization_interface, translation_interface],
    ["Sentiment Analyzer", "Text Summarizer", "Translator"],
    title="AI Multi-Tool (Gradio)"
)

if __name__ == "__main__":
    multi_tool_app.launch()

Writing app.py


## Guía de creación del Space en Hugging Face

### Subtask:
Proporcionar instrucciones detalladas sobre cómo crear un nuevo 'Space' en Hugging Face, elegir el SDK de Gradio y configurar el secreto 'GOOGLE_API_KEY' en la configuración del Space para que la aplicación funcione de forma segura.


### Instrucciones de Despliegue en Hugging Face Spaces

Sigue estos pasos para poner tu aplicación en línea:

1.  **Crear el Space**: Inicia sesión en [Hugging Face](https://huggingface.co/) y haz clic en **'New Space'**.
2.  **Configuración Inicial**:
    *   Dale un nombre a tu Space.
    *   Selecciona **Gradio** como el SDK.
    *   Elige el tipo de licencia y visibilidad (Public/Private).
3.  **Configurar la API Key**:
    *   Una vez creado, ve a la pestaña **'Settings'** de tu Space.
    *   Busca la sección **'Variables and secrets'**.
    *   Haz clic en **'New secret'**.
    *   Nombre: `GOOGLE_API_KEY`.
    *   Valor: (Pega tu clave de API de Gemini).
4.  **Subir Archivos**:
    *   Ve a la pestaña **'Files'**.
    *   Sube el archivo `requirements.txt` que generamos.
    *   Sube el archivo `app.py` que generamos.
5.  **Construcción**: Hugging Face detectará automáticamente los archivos e instalará las librerías. En unos minutos, tu aplicación estará lista en la pestaña 'App'.

## Finalizar tarea

### Subtask:
Resumir los pasos finales para subir los archivos a Hugging Face y verificar que el despliegue se complete correctamente.


## Summary:

### Data Analysis Key Findings

*   **Production File Generation**: Successfully generated two essential files for deployment: `requirements.txt` containing 4 core dependencies and `app.py` consolidating the application logic.
*   **Model Integration**: The application utilizes a hybrid approach, combining the `distilbert-base-uncased-finetuned-sst-2-english` model for sentiment analysis and the `gemini-2.0-flash-lite-001` model for generative summarization.
*   **Security Implementation**: The script was designed to use `os.getenv("GOOGLE_API_KEY")`, ensuring that sensitive credentials are never hardcoded and are instead managed via Hugging Face Secrets.
*   **Multilingual Support**: Integration of `deep-translator` allows the sentiment analysis tool to process non-English text by automatically translating it to English before classification.
*   **User Interface**: A modular Gradio interface was created using `gr.TabbedInterface`, organizing the Sentiment Analyzer, Text Summarizer, and Translator into accessible tabs.

### Insights or Next Steps

*   **Deployment**: The immediate next step is to manually upload `app.py` and `requirements.txt` to a new Hugging Face Space configured with the Gradio SDK.
*   **API Configuration**: Ensure that the `GOOGLE_API_KEY` is added to the "Variables and secrets" section in the Hugging Face Space settings prior to the first build to avoid runtime errors during the Gemini model initialization.
